# 5장 Layout Out Pages

## 5.1 The Layout Tree

지금까지는 `Layout` 클래스가 HTML DOM Tree를 순회하며 화면에 그릴 모든 텍스트의 좌표를 계산했다.

하지만 실제로는 요소마다 레이아웃 방식(블록, 인라인, 플렉스 등)이 다르기 때문에, 레이아웃 트리를 분리한다.



지금까지(Chapter 4)는 단일 `Layout` 클래스가 HTML 트리(DOM)를 순회하며 화면에 그릴 모든 텍스트의 좌표를 계산했습니다. 
하지만 실제 브라우저는 요소마다 레이아웃 방식(블록, 인라인, 플렉스 등)이 다르기 때문에, 이를 하나의 클래스에서 모두 처리하면 코드가 지나치게 복잡해집니다.

이를 해결하기 위해 5장에서는 **HTML 트리(DOM Tree)와 레이아웃 트리(Layout Tree)를 분리**합니다. 

#### 1. 레이아웃 트리의 핵심 개념
- HTML 트리의 노드가 "문서의 구조(의미)"를 나타낸다면, 레이아웃 트리의 노드는 화면에 그려지는 **"시각적인 박스(Box)"** 를 나타냅니다.
- 각 레이아웃 노드는 자신만의 `x`, `y`, `width`, `height`를 가집니다.
- 각 노드는 자신의 자식 노드들에게 "너비(width)"를 알려주고, 자식들은 레이아웃을 마친 뒤 자신의 "높이(height)"를 부모에게 반환하는 재귀적인 구조를 가집니다.

#### 2. 주요 클래스 리팩토링
- **`DocumentLayout`**: 레이아웃 트리의 최상위 루트 노드입니다. 브라우저 창 전체를 나타냅니다.
- **`BlockLayout`**: `<p>`, `<h1>`, `<div>` 등 위에서 아래로 쌓이는 **블록 요소**를 담당하는 레이아웃 노드입니다. 이전 부모(`parent`)와 이전 형제(`previous`) 노드의 참조를 가지고 있어 자신의 Y 좌표를 계산하는 데 사용합니다.

#### 3. 변경된 렌더링 흐름
이전에는 `Layout(nodes).display_list`로 한 번에 처리했지만, 이제는 각 레이아웃 클래스에 있는 `layout()` 메서드를 호출하여 계층적으로 레이아웃을 계산합니다.
```python
# browser.py 변경점 요약
self.document = DocumentLayout(self.nodes)
self.document.layout()
self.display_list = self.document.display_list
```

## 5.2 Block and Inline Layout

HTML 요소는 화면에 배치되는 방식에 따라 크게 **블록(Block)** 요소와 **인라인(Inline)** 요소로 나눌 수 있습니다.

#### 블록 요소 (Block Elements)
- `<p>`, `<div>`, `<h1>` 등
- 부모의 너비를 가득 채우며, 기본적으로 **위에서 아래로(수직으로)** 쌓입니다.
- 즉, 이전 형제 노드(previous sibling)의 아래쪽에 배치됩니다.

#### 인라인 요소 (Inline Elements)
- `<b>`, `<i>`, 일반 텍스트(`Text` 노드) 등
- 텍스트처럼 **왼쪽에서 오른쪽으로(수평으로)** 흐르며, 공간이 부족하면 줄바꿈(wrap)이 일어납니다.

#### `layout_mode` 결정 알고리즘
어떤 요소가 블록 레이아웃을 따를지, 인라인 레이아웃을 따를지는 자식 노드의 구성에 따라 결정됩니다. `layout.py`에 추가된 `layout_mode` 함수가 이 역할을 합니다.

1. **텍스트 노드인 경우**: 항상 "inline"
2. **자식 중 하나라도 블록 요소가 있는 경우**: 이 노드도 "block" 레이아웃을 사용해야 함 (블록 요소가 섞여 있으면 전체를 블록 컨텍스트로 처리)
3. **자식이 모두 인라인 요소(또는 텍스트)인 경우**: "inline" 레이아웃 사용
4. **자식이 없는 경우**: 기본적으로 "block"

#### BlockLayout의 `layout` 동작 분기
`BlockLayout`은 `layout_mode`의 결과에 따라 다르게 동작합니다.
- `mode == "block"`: 자식 요소들도 별도의 블록으로 간주하여 재귀적으로 `BlockLayout` 객체를 생성하고 레이아웃 트리에 추가합니다.
- `mode == "inline"`: 자식 요소들이 인라인 컨텍스트를 구성하므로, 이전 챕터에서 사용하던 방식(`recurse`, `word`, `flush` 등)을 사용하여 텍스트 라인 브레이킹 및 인라인 배치를 수행합니다. (현재 코드는 임시로 `BlockLayout` 내부에 `recurse` 로직을 남겨둔 상태입니다.)

### `print_tree`를 활용한 DOM 트리 vs 레이아웃 트리 비교
`html.py`에 정의된 `print_tree` 함수를 사용하면 트리의 구조를 직관적으로 확인할 수 있습니다.
- HTML 트리는 `Element`와 `Text` 노드로 구성됩니다.
- 레이아웃 트리는 `DocumentLayout`을 루트로 하고, 그 아래에 `BlockLayout` (또는 나중에 추가될 `InlineLayout`) 노드들로 구성됩니다.

아래 코드를 실행하여 두 트리가 어떻게 다르게 구성되는지 확인해 보세요!

In [1]:
from browser.html import HTMLParser, print_tree
from browser.layout import DocumentLayout

html_content = """
<html>
  <body>
    <h1>Hello, Layout Tree!</h1>
    <p>This is a <b>test</b> of the layout tree.</p>
  </body>
</html>
"""

print("=== HTML DOM Tree ===")
nodes = HTMLParser(html_content).parse()
print_tree(nodes)

print("\n=== Layout Tree ===")
document = DocumentLayout(nodes)
document.layout()
print_tree(document)


=== HTML DOM Tree ===
 <html>
   <body>
     <h1>
       'Hello, Layout Tree!'
     <p>
       'This is a '
       <b>
         'test'
       ' of the layout tree.'

=== Layout Tree ===
 DocumentLayout()
   BlockLayout(<html>)
     BlockLayout(<body>)
       BlockLayout(<h1>)
       BlockLayout(<p>)


## 5.3 Size and Position (크기와 위치)

레이아웃 트리의 핵심은 각 `BlockLayout` 요소가 자신만의 좌표(`x`, `y`)와 크기(`width`, `height`)를 계산하는 것입니다. 이 계산은 **위에서 아래로(Top-Down)** 흐르는 성질과 **아래에서 위로(Bottom-Up)** 흐르는 성질이 결합되어 이루어집니다.

### 1. Width와 위치 (Top-Down)
- **`x` 좌표**: 부모의 `x` 좌표를 그대로 상속받습니다.
- **`width`**: 부모가 제공하는 가용 너비(부모의 `width`)를 꽉 채우도록 설정됩니다.
- **`y` 좌표**: 부모의 `y`부터 시작하되, 이전 형제 노드(`previous`)가 있다면 그 형제 노드의 아래쪽(`previous.y + previous.height`)에서부터 시작합니다.
- **왜 먼저 계산될까요?** 텍스트를 몇 줄로 줄바꿈(Word Wrap)할지 결정하려면 상자의 너비(Width)를 먼저 확정해야 하기 때문입니다.

### 2. Height (Bottom-Up)
- 블록 요소의 높이는 고정되어 있지 않고, 내부의 자식 요소들이 얼마나 많은 공간을 차지하느냐에 따라 늘어납니다.
- 따라서 부모는 자식들(`self.children`)의 `layout()`을 모두 호출하여 배치를 끝낸 뒤에야 높이를 확정할 수 있습니다.
- **블록 모드인 경우**: 모든 자식들의 높이(`child.height`)를 합산하여 자신의 높이로 확정합니다.
- **인라인 모드인 경우**: 텍스트를 모두 줄바꿈하여 배치한 뒤의 최종 커서 위치(`self.cursor_y`)가 곧 텍스트 상자의 높이가 됩니다.

```python
# BlockLayout.layout() 내의 높이 계산 로직
if mode == "block":
    self.height = sum([child.height for child in self.children])
else:
    self.height = self.cursor_y
```